In [ ]:
import pandas as pd
import numpy as np
# import pyterrier_deepct
import pyterrier as pt
if not pt.started():
  pt.init(boot_packages=["com.github.terrierteam:terrier-prf:-SNAPSHOT"])

from xml.etree import ElementTree
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics2020.xml')
data = tree.getroot()
titles20 = []
titles21 = []
title_ids20 = []
title_ids21 = []
nums20 = []
nums21 = []
for topic in data:
    titles20.append(topic.find('title').text.lower()[:-1])
    num20 = topic.find('number').text
    nums20.append(num20)
tree = ElementTree.parse('/workspace/src/evaluate_targers/topics.xml')
data = tree.getroot()    
for topic in data:
    titles21.append(topic.find('title').text.lower()[:-1])
    num21 = topic.find('number').text
    nums21.append(num21)
topics = pd.DataFrame()
topics['qid'] = nums20 + nums21
topics['query'] = titles20 + titles21

topics21 = pd.DataFrame()
topics21['qid'] = nums21
topics21['query'] = titles21

topics20 = pd.DataFrame()
topics20['qid'] = nums20
topics20['query'] = titles20

qrels_path=("/workspace/src/evaluate_targers/all_qrels.qrels")
qrels = pt.io.read_qrels(qrels_path)

qrels21_path=("/workspace/src/evaluate_targers/qrels2021.qrels")
qrels21 = pt.io.read_qrels(qrels21_path)

qrels20_path=("/workspace/src/evaluate_targers/qrels2020.qrels")
qrels20 = pt.io.read_qrels(qrels20_path)

docnos = qrels.docno
docnos21 = qrels21.docno
docnos20 = qrels20.docno

In [ ]:
import os
# rootdir = "/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/DeepCT_indexes/"
rootdir = "/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/targer_indexes/pd_index_original-100"

objective="ndcg_cut_5"
for i, (subdir, dirs, files) in enumerate(os.walk(rootdir)):
    for file in files:
        if(file == 'data.properties'):
            path = subdir + '/'+ file
            print('\n' + subdir)
            indexref = pt.IndexRef.of(path)
            index = pt.IndexFactory.of(indexref)
#             bm25_rm3 = (pt.BatchRetrieve(indexref, wmodel="BM25") >> 
#             pt.rewrite.RM3(indexref) >> 
#             pt.BatchRetrieve(indexref, wmodel="BM25"))
            bm25_for_qe = pt.BatchRetrieve(index, wmodel="BM25")
            rm3 = pt.rewrite.RM3(index, fb_terms=10, fb_docs=3)
            pipe_qe = bm25_for_qe >> rm3 >> bm25_for_qe
            
            param_map = {
                rm3 : {
                    "fb_terms" : list(range(4, 16, 2)),
                    "fb_docs" : list(range(4, 10, 2)),
                    "fb_lambda": [0.2,0.4,0.6,0.8,1.0]
                }
            }
            
            pipe_qe, parameters = pt.KFoldGridSearch(
                pipe_qe,
                param_map,
                [topics20, topics21],
                [qrels20, qrels21],
                objective
            )
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics, qrels, 
                                names=['tuned_bm25_rm3','orignal_bm25_rm3'], 
                                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics20, qrels, names=['20original_tuned_bm25_rm3', '20orignal_bm25_rm3'], 
                                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics21, qrels, names=['21original_tuned_bm25_rm3', '21orignal_bm25_rm3'], 
                                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            pipe_qe.to_csv(path_or_buf=f"/workspace/src/grid_search_right_qrels/final_right/run/original_tuned_bm25_rm3.csv")
            

In [ ]:
import os
rootdir = "/mnt/ceph/storage/data-in-progress/data-teaching/theses/wstud-thesis-hollatz/DeepCT_indexes/"
folders = ['conclusions','pool_topics','topics','topics_conclusion','topics_conclusion_leakage','topics_leakage','pool_topics_conclusions','conclusions_leakage','pools_conclusions']
objective="ndcg_cut_5"
i = 0
for subdir, dirs, files in os.walk(rootdir):
    for file in files:
        if(file == 'data.properties'):
            path = subdir + '/'+ file
            print('\n' + subdir)
            indexref = pt.IndexRef.of(path)
            index = pt.IndexFactory.of(indexref)
            bm25_for_qe = pt.BatchRetrieve(index, wmodel="BM25")
            rm3 = pt.rewrite.RM3(index, fb_terms=10, fb_docs=3)
            pipe_qe = bm25_for_qe >> rm3 >> bm25_for_qe
            
            param_map = {
                rm3 : {
                    "fb_terms" : list(range(4, 16, 2)),
                    "fb_docs" : list(range(4, 10, 2)),
                    "fb_lambda": [0.2,0.4,0.6,0.8,1.0]
                }
            }
            
            pipe_qe, parameters = pt.KFoldGridSearch(
                pipe_qe,
                param_map,
                [topics20, topics21],
                [qrels20, qrels21],
                objective
            )
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics, qrels, names=[folders[i] + '_tuned_bm_rm3', folders[i] + '_orignal_bm_rm3'],
                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics20, qrels, names=['20' + folders[i] + 'tuned_bm_rm3', '20' + folders[i] + '_bm_rm3'], 
                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics21, qrels, names=['21' + folders[i] + 'tuned_bm_rm3', '21'+ folders[i] + '_bm_rm3'], 
                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            
            pipe_qe.to_csv(path_or_buf=f"/workspace/src/grid_search_right_qrels/final_right/run/{folders[i]}_tuned_bm_rm3.csv")
            i+=1

In [ ]:
import os
rootdir = "/workspace/src/grid_search_right_qrels/super_folder/"
folders = ['marco-pretrained']
objective="ndcg_cut_5"
i = 0
for subdir, dirs, files in os.walk(rootdir):
    for file in files:
        if(file == 'data.properties'):
            path = subdir + '/'+ file
            print('\n' + subdir)
            indexref = pt.IndexRef.of(path)
            index = pt.IndexFactory.of(indexref)
            bm25_for_qe = pt.BatchRetrieve(index, wmodel="BM25")
            rm3 = pt.rewrite.RM3(index, fb_terms=10, fb_docs=3)
            pipe_qe = bm25_for_qe >> rm3 >> bm25_for_qe
            
            param_map = {
                rm3 : {
                    "fb_terms" : list(range(4, 16, 2)),
                    "fb_docs" : list(range(4, 10, 2)),
                    "fb_lambda": [0.2,0.4,0.6,0.8,1.0]
                }
            }
            
            pipe_qe, parameters = pt.KFoldGridSearch(
                pipe_qe,
                param_map,
                [topics20, topics21],
                [qrels20, qrels21],
                objective
            )
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics, qrels, names=[folders[i] + '_tuned_bm_rm3', folders[i] + '_orignal_bm_rm3'],
                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics20, qrels, names=['20' + folders[i] + 'tuned_bm_rm3', '20' + folders[i] + '_bm_rm3'], 
                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            
            print(pt.Experiment([pipe_qe, bm25_for_qe], topics21, qrels, names=['21' + folders[i] + 'tuned_bm_rm3', '21'+ folders[i] + '_bm_rm3'], 
                eval_metrics=["ndcg_cut_5", "ndcg_cut_25", "bpref"]).to_latex())
            pipe_qe.to_csv(path_or_buf=f"/workspace/src/grid_search_right_qrels/final_right/run/{folders[i]}_tuned_bm_rm3.csv")
#             pipe_qe.to_csv(path_or_buf=f"/workspace/src/runs_bm25_rm3/{folders[i]}_tuned_bm_rm3.csv")
            i+=1